In [1]:
import datetime
import logging  # 追加：ログ制御用
import warnings  # 追加：警告非表示用
import pandas as pd
import yfinance as yf
from IPython.display import display, HTML

# 警告メッセージ（FutureWarningなど）を非表示にする
warnings.filterwarnings("ignore")

# yfinanceの内部エラー出力を非表示にする
yf_logger = logging.getLogger('yfinance')
yf_logger.setLevel(logging.CRITICAL)

# Define the lookback periods (in business days) for performance calculation.
LOOKBACK_DAYS = {
    "1 Day (%)": -2,
    "3 Days (%)": -4,
    "1 Week (%)": -6,
    "1 Month (%)": -21,
    "6 Months (%)": -126,
    "1 Year (%)": -252,
}

# =====================================================================
# DATA PROCESSING FUNCTION (Supports Market Segmentation)
# =====================================================================
def show_market_performances(market_dict):
    today = datetime.date.today()
    start_date = today - datetime.timedelta(days=60)
    
    # Format numerical values to 2 decimal places
    format_config = {"Latest Price": "{:,.2f}"}
    for column_name in LOOKBACK_DAYS.keys():
        format_config[column_name] = "{:+.2f}%"
        
    # ハイライト用の条件関数
    def highlight_extreme_values(val):
        if pd.isna(val):
            return ''
        if val >= 10:
            return 'background-color: #ffcccc; color: #cc0000; font-weight: bold;'
        elif val <= -10:
            return 'background-color: #cce5ff; color: #004085; font-weight: bold;'
        return ''
        
    # Process each market section
    for market_name, ticker_dict in market_dict.items():
        records = []
        
        # Display market header
        display(HTML(f"<h3 style='margin-top: 20px; color: #333;'>■ {market_name} Market</h3>"))
        
        for name, ticker in ticker_dict.items():
            try:
                # 【修正】 ignore_tz=True を追加してタイムゾーン警告を抑制
                df = yf.download(ticker, start=start_date, end=today, progress=False, ignore_tz=True)
                
                if df.empty:
                    # 元のコードにあったWarning表示も邪魔ならコメントアウトしてください
                    # print(f"[Warning] No data found for: {name} ({ticker})")
                    continue
                    
                prices = df["Close"].squeeze()
                latest_price = prices.iloc[-1]
                
                row = {
                    "Name": name,
                    "Ticker": ticker,
                    "Latest Price": latest_price
                }
                
                for label, index in LOOKBACK_DAYS.items():
                    if len(prices) >= abs(index):
                        past_price = prices.iloc[index]
                    else:
                        past_price = prices.iloc[0]
                    
                    row[label] = ((latest_price - past_price) / past_price) * 100
                    
                records.append(row)
                
            except Exception as e:
                # エラー時のprintも非表示にしたい場合はコメントアウトしてください
                pass
        
        if records:
            df_performance = pd.DataFrame(records)
            
            # .map を使用
            styled_df = (df_performance.style
                         .format(format_config)
                         .map(highlight_extreme_values, subset=list(LOOKBACK_DAYS.keys()))
                         .hide(axis="index"))
            
            display(styled_df)
        else:
            print("No data available for this market.")

In [2]:
# =====================================================================
# CONFIGURATION & EXECUTION (Segmented by Market)
# =====================================================================
MARKET_TICKERS = {
    "United States": {
        "S&P 500": "^GSPC",
        "NASDAQ Composite": "^IXIC",
        "Apple": "AAPL",
        "NVIDIA": "NVDA",
        "Microsoft": "MSFT",
    },
    "Japan": {
        "Nikkei 225": "^N225",
        "Toyota Motor": "7203.T",
        "Sony Group": "6758.T",
        "Mitsubishi UFJ Financial": "8306.T",
    },
    "Singapore": {
        "STI Index": "^STI",
        "DBS Group": "D05.SI",
        "UOB": "U11.SI",
        "Singtel": "Z74.SI",
        "CapitaLand Ascendas REIT": "A17U.SI",
    },
    "Japan Stock": {
        "小松製作所 (6301)": "6301.T",
        "ニデック (6594)": "6594.T",
        "未来工業 (7931)": "7931.T",
        "東部ネットワーク (9036)": "9036.T",
        "ニトリHD (9843)": "9843.T",
        "MRK HD (9980)": "9980.T",
    },
    "US Stock (ETF)": {
        "iシェアーズ コア米国総合債券ETF (AGG)": "AGG"
    }
}

# Run and display the segmented tables
show_market_performances(MARKET_TICKERS)

Name,Ticker,Latest Price,1 Day (%),3 Days (%),1 Week (%),1 Month (%),6 Months (%),1 Year (%)
S&P 500,^GSPC,"7,572.40",+0.38%,-0.04%,+1.20%,+0.24%,+2.29%,+2.29%
NASDAQ Composite,^IXIC,"26,269.23",+0.62%,-0.05%,+1.54%,-1.55%,+0.68%,+0.68%
Apple,AAPL,327.50,+4.01%,+3.86%,+4.50%,+10.49%,+9.96%,+9.96%
NVIDIA,NVDA,212.50,+0.33%,+0.73%,+4.11%,+0.02%,-4.31%,-4.31%
Microsoft,MSFT,395.63,+2.78%,+2.73%,+3.21%,-1.03%,-6.39%,-6.39%


Name,Ticker,Latest Price,1 Day (%),3 Days (%),1 Week (%),1 Month (%),6 Months (%),1 Year (%)
Nikkei 225,^N225,"68,751.51",+1.49%,+0.28%,+2.89%,-1.65%,+13.05%,+13.05%
Toyota Motor,7203.T,"2,875.50",+1.29%,+1.86%,-0.47%,+2.33%,-2.67%,-2.67%
Sony Group,6758.T,"3,389.00",+0.18%,+0.89%,-2.28%,+3.13%,-5.76%,-5.76%
Mitsubishi UFJ Financial,8306.T,"3,697.00",+2.69%,+6.82%,+7.69%,+12.99%,+23.44%,+23.44%


Name,Ticker,Latest Price,1 Day (%),3 Days (%),1 Week (%),1 Month (%),6 Months (%),1 Year (%)
STI Index,^STI,"5,559.72",+1.17%,+1.65%,+3.54%,+7.40%,+11.27%,+11.27%
DBS Group,D05.SI,72.98,+1.32%,+3.59%,+5.62%,+12.26%,+20.11%,+20.11%
UOB,U11.SI,44.95,+1.28%,+1.28%,+3.74%,+14.23%,+20.54%,+20.54%
Singtel,Z74.SI,4.41,-0.68%,+0.23%,+0.92%,+1.38%,-9.45%,-9.45%
CapitaLand Ascendas REIT,A17U.SI,2.49,+0.00%,-1.19%,+0.00%,-3.11%,+2.05%,+2.05%


Name,Ticker,Latest Price,1 Day (%),3 Days (%),1 Week (%),1 Month (%),6 Months (%),1 Year (%)
小松製作所 (6301),6301.T,"6,420.00",+0.85%,+0.39%,-1.61%,-3.46%,+1.55%,+1.55%
ニデック (6594),6594.T,"2,626.00",+0.69%,-1.28%,+1.66%,-6.08%,+3.92%,+3.92%
未来工業 (7931),7931.T,"3,290.00",+1.54%,+2.33%,+2.97%,+7.34%,+11.00%,+11.00%
東部ネットワーク (9036),9036.T,"1,314.00",+6.40%,+3.87%,+3.46%,+9.68%,+1.00%,+1.00%
ニトリHD (9843),9843.T,"2,403.00",-2.02%,+0.97%,-1.62%,-7.09%,-0.97%,-0.97%
MRK HD (9980),9980.T,95.00,+1.06%,-1.04%,-1.04%,-1.04%,-3.06%,-3.06%


Name,Ticker,Latest Price,1 Day (%),3 Days (%),1 Week (%),1 Month (%),6 Months (%),1 Year (%)
iシェアーズ コア米国総合債券ETF (AGG),AGG,98.14,+0.14%,+0.06%,+0.10%,-0.39%,+0.81%,+0.81%


In [3]:
MARKET_TICKERS = {
    "Singapore Healthcare": {
        # --- 病院経営・総合クリニック ---
        "Raffles Medical Group (ラッフルズ医療)": "BSL.SI",  # 【修正】R01.SI から変更
        "IHH Healthcare (マウント・エリザベス等経営)": "Q0F.SI",  # 【修正】I01.SI から変更
        "Thomson Medical Group (トムソン医療)": "A50.SI",
        
        # --- 医療不動産（REIT / 病院・介護施設） ---
        "Parkway Life REIT (パークウェイ・ライフ)": "C2PU.SI",
        "First REIT (ファースト・リート)": "AW9U.SI",
        
        # --- 専門医療・歯科クリニックチェーン ---
        "Q&M Dental Group (歯科大手チェーン)": "QC7.SI",
        "Alliance Healthcare Group (総合診療・クリニック)": "MIJ.SI",  # 【修正】1D8から変更
        "Medinex Limited (医療サポート・クリニック運営)": "OTX.SI",
        
        # --- 医療用消耗品・製薬 ---
        "Medtecs International (医療用消耗品)": "546.SI",
        "Hyphens Pharma (医薬品・皮膚科製品)": "1J5.SI",
        "Riverstone": "AP4.SI"
    }
}
show_market_performances(MARKET_TICKERS)

Name,Ticker,Latest Price,1 Day (%),3 Days (%),1 Week (%),1 Month (%),6 Months (%),1 Year (%)
Raffles Medical Group (ラッフルズ医療),BSL.SI,0.94,+0.00%,-0.53%,+2.73%,-1.05%,-1.57%,-1.57%
IHH Healthcare (マウント・エリザベス等経営),Q0F.SI,2.75,-2.83%,+1.85%,-3.51%,-0.72%,-6.14%,-6.14%
Thomson Medical Group (トムソン医療),A50.SI,0.05,+0.00%,+0.00%,+1.89%,-1.82%,-5.26%,-5.26%
Parkway Life REIT (パークウェイ・ライフ),C2PU.SI,4.15,+0.48%,+0.97%,+0.97%,+4.80%,+4.80%,+4.80%
First REIT (ファースト・リート),AW9U.SI,0.22,+0.00%,-2.17%,+0.00%,-2.17%,-4.26%,-4.26%
Q&M Dental Group (歯科大手チェーン),QC7.SI,0.54,-0.92%,-2.70%,-3.57%,-10.00%,-10.74%,-10.74%
Alliance Healthcare Group (総合診療・クリニック),MIJ.SI,0.16,+0.00%,+0.00%,+0.00%,-0.63%,+1.94%,+1.94%
Medinex Limited (医療サポート・クリニック運営),OTX.SI,0.22,+0.00%,-2.17%,-2.17%,+0.00%,+2.27%,+2.27%
Medtecs International (医療用消耗品),546.SI,0.11,+0.88%,+0.00%,+0.00%,-5.00%,-7.32%,-7.32%
Hyphens Pharma (医薬品・皮膚科製品),1J5.SI,0.35,+0.00%,+0.00%,+0.00%,+2.94%,+11.11%,+11.11%


In [4]:
MARKET_TICKERS = {
"Singapore Food & Agribusiness": {
        "Wilmar International (ウィルマー・世界最大級のパーム油)": "F34.SI",
        "Olam Group (オラム・ココア、コーヒー、ナッツ等大手)": "VC2.SI",
        "Riverstone": "AP4.SI",
        "ヤンジジャン・シップビルディング(BS6)": "BS6.SI",
        "Sembcorp Ind (U96)": "U96.SI",
    }
}
show_market_performances(MARKET_TICKERS)

Name,Ticker,Latest Price,1 Day (%),3 Days (%),1 Week (%),1 Month (%),6 Months (%),1 Year (%)
Wilmar International (ウィルマー・世界最大級のパーム油),F34.SI,3.91,+0.51%,+1.56%,+3.17%,+8.01%,+5.39%,+5.39%
Olam Group (オラム・ココア、コーヒー、ナッツ等大手),VC2.SI,1.25,+2.46%,+1.63%,+5.04%,-3.85%,+6.84%,+6.84%
Riverstone,AP4.SI,0.86,+0.58%,+0.58%,+4.88%,+1.18%,-5.49%,-5.49%
ヤンジジャン・シップビルディング(BS6),BS6.SI,3.65,+2.24%,+1.11%,+6.41%,+0.00%,-6.89%,-6.89%
Sembcorp Ind (U96),U96.SI,5.57,+1.09%,-1.94%,-1.59%,-11.73%,-8.84%,-8.84%


<div style="background-color: #EBCB8B; color: #2E3440; padding: 12px; border-radius: 6px; font-weight: bold;">
</div>